# M3 on RunPod: executor RL with poset vs chain vs flat credit

The real-LLM test of Theorem 1. The planner is the frozen base model (Qwen3-1.7B, thinking mode, budget forcing, as in M2 run 2). Plans are made once and cached, so every method trains on **the same plans and problems in the same order**; methods differ only in the per-step advantage:

* `poset`: a step is credited if it and its prerequisites verified (verified ideal)
* `chain`: a step is credited only if every step before it in the written order verified
* `flat`: outcome-only GRPO

Evaluation is stratified by plan width: **wide plans (width >= 2) are the test, chains (width 1) are the control.** Theorem 1 predicts poset > chain on wide plans and no difference on chains.

**Costs money while the pod runs.** Stop the pod (RunPod console -> Stop) whenever you are not running a cell. Everything is saved to `/workspace`, which survives a stop.

Order: cells 1-3, restart the kernel, cells 3-8 (cell 8 is a 10-minute smoke test), then cell 9 (the long run), 10, 11.

In [ ]:
# 1. GPU check: expect an A100 80GB (or RTX PRO 6000)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install (a few minutes). Then: Kernel -> Restart Kernel, and continue from cell 3.
!pip install -q vllm peft datasets sympy antlr4-python3-runtime==4.11

In [ ]:
# 3. Paths and code. Upload pohrl.zip to /workspace/pohrl/ first (JupyterLab file browser, left).
import os, sys
BASE = '/workspace/pohrl'
os.makedirs(BASE, exist_ok=True)
os.environ['HF_HOME'] = '/workspace/hf_cache'      # model + datasets cached on the volume
!cd {BASE} && unzip -oq pohrl.zip -d code
sys.path.insert(0, f'{BASE}/code/pohrl')
import pohrl, pohrl_llm; print('code ok')

In [ ]:
# 4. Configuration
MODEL = 'Qwen/Qwen3-1.7B'
METHODS = ['poset', 'chain', 'flat']
SEEDS = [0]                  # add 1 later if budget allows
STEPS, PROBLEMS_PER_STEP, G = 150, 16, 8
EVAL_EVERY = 25
N_TRAIN = 1200               # MATH train, levels 3-5
EVAL_PER_BUCKET = 50         # up to 50 eval problems per width bucket (1, 2-3, 4+)
GPU_UTIL = 0.35              # share of GPU memory for vLLM; the trainer uses the rest
OUT = f'{BASE}/m3_{MODEL.split("/")[-1]}'
os.makedirs(OUT, exist_ok=True)

In [ ]:
# 5. Data: MATH train (levels 3-5) for RL; MATH500 levels 3-5 for evaluation (disjoint splits)
import random
from pohrl_llm import data as D
SUBJECTS = ['algebra', 'counting_and_probability', 'geometry', 'intermediate_algebra',
            'number_theory', 'prealgebra', 'precalculus']
keep = lambda r: str(r.get('level', '')) in ('Level 3', 'Level 4', 'Level 5')
train = []
try:
    for s in SUBJECTS:
        train += D.load_hf('EleutherAI/hendrycks_math', split='train', qkey='problem', akey='solution', config=s, filter_fn=keep)
except Exception as e:
    print('hendrycks_math failed (', type(e).__name__, '), trying MATH-lighteval')
    train = D.load_hf('DigitalLearningGmbH/MATH-lighteval', split='train', qkey='problem', akey='solution', filter_fn=keep)
random.Random(0).shuffle(train); train = train[:N_TRAIN]
for i, r in enumerate(train): r['id'] = f'tr{i}'
ev_pool = [r for r in D.load_hf('math500') if int(r.get('level', 0)) >= 3]
print(len(train), 'train,', len(ev_pool), 'eval pool')

In [ ]:
# 6. vLLM with LoRA (shares the GPU with the trainer)
from pohrl_llm.backends import VLLMBackend
from pohrl_llm import prompts as PR
backend = VLLMBackend(MODEL, dtype='bfloat16', gpu_memory_utilization=GPU_UTIL, enable_lora=True,
                      max_lora_rank=64, max_model_len=8192)
render = lambda msgs: PR.chat(msgs, backend.tokenizer, enable_thinking=False)

In [ ]:
# 7. Plans from the frozen model, cached in /workspace (shared by all methods). Roughly 20-40 minutes the first time.
import collections
from pohrl_llm import run_m3
train_all = run_m3.cache_plans(train, backend, render, f'{OUT}/plans_train.jsonl')
ev_all = run_m3.cache_plans(ev_pool, backend, render, f'{OUT}/plans_eval.jsonl')
train_planned = run_m3.planned_only(train_all)
eval_planned = run_m3.stratify(ev_all, EVAL_PER_BUCKET)
cnt = lambda xs: dict(collections.Counter(run_m3.width_bucket(p.plan.poset.width()) for p in xs))
print('train fallback rate:', round(1 - len(train_planned) / len(train_all), 3), '| train widths:', cnt(train_planned))
print('eval widths:', cnt(eval_planned), '| total eval:', len(eval_planned))

In [ ]:
# 8. SMOKE TEST (about 10 minutes): 2 RL steps on 2 problems, tiny evaluation. Checks the trainer,
#    adapter saving and vLLM loading the adapter, before the long run. Separate folder, so nothing mixes.
from pohrl_llm.trainer import LoRATrainer
import gc, torch, json
factory = lambda run_dir, resume: LoRATrainer(MODEL, run_dir, lr=1e-5, rank=64, resume=resume, micro_bs=2)
smoke = run_m3.train('poset', 0, train_planned[:8], eval_planned[:6], backend, render, factory, f'{OUT}/smoke',
                     steps=2, problems_per_step=2, G=2, eval_every=2, verify_mode='consensus')
for r in smoke: print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items()})
gc.collect(); torch.cuda.empty_cache()
print('peak GPU memory (GB):', round(torch.cuda.max_memory_allocated() / 1e9, 1))

In [ ]:
# 9. FULL RUN (resumable: if the pod stops, start it again, run cells 3-7, then this cell).
#    Roughly 6-8 hours per method on an A100 80GB; methods run one after another.
logs = {}
for seed in SEEDS:
    for m in METHODS:
        logs[(m, seed)] = run_m3.train(m, seed, train_planned, eval_planned, backend, render, factory, OUT,
                                       steps=STEPS, problems_per_step=PROBLEMS_PER_STEP, G=G,
                                       eval_every=EVAL_EVERY, verify_mode='consensus')
        gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 10. Curves: overall, chains (control), width 2-3 and 4+ (test)
import matplotlib.pyplot as plt, json, glob
cols = {'flat': '#999', 'chain': '#d9822b', 'poset': '#2a6fdb'}
fig, ax = plt.subplots(1, 4, figsize=(16, 3.5))
for path in sorted(glob.glob(f'{OUT}/*_s*/state.json')):
    name = path.split('/')[-2]; m = name.split('_s')[0]
    ev_rows = [r for r in json.load(open(path))['log'] if 'acc' in r]
    for k, key in enumerate(['acc', 'acc_w1', 'acc_w2-3', 'acc_w4+']):
        xs = [r['step'] for r in ev_rows if r.get(key) is not None]
        ax[k].plot(xs, [r[key] for r in ev_rows if r.get(key) is not None], color=cols.get(m, 'k'), label=name)
        ax[k].set_title({'acc': 'all', 'acc_w1': 'chains (control)', 'acc_w2-3': 'width 2-3 (test)', 'acc_w4+': 'width 4+ (test)'}[key])
        ax[k].set_xlabel('RL step')
ax[0].legend(); plt.tight_layout(); plt.savefig(f'{OUT}/m3_curves.png', dpi=150)

In [ ]:
# 11. Package logs and cached plans (no model weights) for download: right-click the zip in the file browser -> Download
!cd {OUT} && zip -qr {BASE}/m3_logs.zip */state.json plans_*.jsonl m3_curves.png && ls -la {BASE}/m3_logs.zip